In [57]:
import pandas as pd
import numpy as np
from datetime import datetime

# -----------------------
# LOAD DATA
# -----------------------

df = pd.read_csv("/Users/benvarvill/Downloads/attendees.csv")

# -----------------------
# CLEANING
# -----------------------

df["Name_clean"] = df["Name"].str.lower().str.strip()
df["Institution_clean"] = df["Institution"].str.lower().str.strip()
df["Country_clean"] = df["Country"].str.lower().str.strip()

# Ensure numeric
df["Appearances"] = pd.to_numeric(df["Appearances"], errors="coerce").fillna(0)

# -----------------------
# DEDUPLICATE
# Keep the row with highest appearances
# -----------------------

df = df.sort_values("Appearances", ascending=False)

df_deduped = df.drop_duplicates(
    subset=["Name_clean", "Country_clean"],
    keep="first"
)

print("After dedupe:", len(df_deduped))
df = df[~df["Title"].str.contains("student", case=False, na=False)]


After dedupe: 20511


In [59]:
df_deduped

,Country,State,Institution,Name,Title,Appearances,Years,Unnamed: 7,Name_clean,Institution_clean,Country_clean
18828,United States,TX,Sunovion,Michael Barber,MSL,11,"2014, 2014, 2015, 2015, 2016, 2016, 2017, 2017...",NaN,michael barber,sunovion,united states
18150,United States,SC,42-South Carolina Psychiatric Association,Eric Williams,Associate Dean of Student Affairs,9,"2014, 2015, 2016, 2016, 2017, 2023, 2024, 2024...",NaN,eric williams,42-south carolina psychiatric association,united states
1995,United States,CA,UC San Diego,Sidney Zisook,Professor,8,"2014, 2015, 2016, 2017, 2023, 2024, 2025, 2025",NaN,sidney zisook,uc san diego,united states
18149,United States,SC,33-Washington State Psychiatric Association,James Lee,Medical Director,8,"2014, 2017, 2023, 2023, 2024, 2024, 2025, 2025",NaN,james lee,33-washington state psychiatric association,united states
17147,United States,PA,Georgetown University,David Goldstein,Psychiatrist,8,"2014, 2014, 2015, 2015, 2016, 2023, 2024, 2025",NaN,david goldstein,georgetown university,united states
...,...,...,...,...,...,...,...,...,...,...,...
8015,United States,MA,Sage/Biogen,Sarah Martini,NaN,1,2023,NaN,sarah martini,sage/biogen,united states
8016,United States,MA,Sage/Biogen,Sean Murphy,NaN,1,2023,NaN,sean murphy,sage/biogen,united states
8017,United States,MA,Sage/Biogen,Sibin Stephen,Senior Medical Science Liaison,1,2023,NaN,sibin stephen,sage/biogen,united states
8018,United States,MA,Sage/Biogen,Sophia Yu,"Associate Director, Marketing",1,2023,NaN,sophia yu,sage/biogen,united states


In [61]:
def frequency_score(times):
    if times >= 8:
        return 5
    elif times >= 5:
        return 4
    elif times >= 3:
        return 3
    elif times == 2:
        return 2
    elif times == 1:
        return 1
    else:
        return 0

from datetime import datetime

current_year = datetime.now().year

def recency_score(years_string):
    if pd.isna(years_string):
        return 0
    
    try:
        years = [int(y) for y in str(years_string).split(",")]
        most_recent = max(years)
        gap = current_year - most_recent
        
        if gap == 0:
            return 5   # attended this year
        elif gap == 1:
            return 4
        elif gap == 2:
            return 3
        elif gap <= 4:
            return 2
        elif gap <= 7:
            return 1
        else:
            return 0
    except:
        return 0

def momentum_score(years_string):
    if pd.isna(years_string):
        return 0
    
    try:
        years = sorted([int(y) for y in str(years_string).split(",")])
        current_year = datetime.now().year
        
        recent_years = [y for y in years if y >= current_year - 3]
        
        if len(recent_years) >= 3:
            return 3
        elif len(recent_years) == 2:
            return 2
        elif len(recent_years) == 1:
            return 1
        else:
            return 0
    except:
        return 0

def authority_score(title):
    if pd.isna(title):
        return 1
    
    title = title.lower()

    tier1 = ["ceo", "cto", "coo", "president", "founder",
             "managing director", "executive director"]
    
    tier2 = ["director", "head", "chair", "dean",
             "vice president", "vp", "chief scientist"]
    
    tier3 = ["principal investigator", "professor",
             "partner", "senior fellow",
             "lab lead", "program manager"]
    
    tier4 = ["researcher", "scientist", "engineer", "associate"]

    for word in tier1:
        if word in title:
            return 5

    for word in tier2:
        if word in title:
            return 4

    for word in tier3:
        if word in title:
            return 3

    for word in tier4:
        if word in title:
            return 2

    return 1
    
df["FrequencyScore"] = df["Appearances"].apply(frequency_score)
df["RecencyScore"] = df["Years"].apply(recency_score)
df["MomentumScore"] = df["Years"].apply(momentum_score)
df["AuthorityScore"] = df["Title"].apply(authority_score)


df["FinalPreScore"] = (
    df["RecencyScore"] * 0.45 +
    df["FrequencyScore"] * 0.25 +
    df["MomentumScore"] * 0.10 +
    df["AuthorityScore"] * 0.20
)


In [63]:
df

,Country,State,Institution,Name,Title,Appearances,Years,Unnamed: 7,Name_clean,Institution_clean,Country_clean,FrequencyScore,RecencyScore,MomentumScore,AuthorityScore,FinalPreScore
18828,United States,TX,Sunovion,Michael Barber,MSL,11,"2014, 2014, 2015, 2015, 2016, 2016, 2017, 2017...",NaN,michael barber,sunovion,united states,5,3,3,1,3.10
1995,United States,CA,UC San Diego,Sidney Zisook,Professor,8,"2014, 2015, 2016, 2017, 2023, 2024, 2025, 2025",NaN,sidney zisook,uc san diego,united states,5,4,3,3,3.95
18149,United States,SC,33-Washington State Psychiatric Association,James Lee,Medical Director,8,"2014, 2017, 2023, 2023, 2024, 2024, 2025, 2025",NaN,james lee,33-washington state psychiatric association,united states,5,4,3,5,4.35
17147,United States,PA,Georgetown University,David Goldstein,Psychiatrist,8,"2014, 2014, 2015, 2015, 2016, 2023, 2024, 2025",NaN,david goldstein,georgetown university,united states,5,4,3,1,3.55
9814,United States,MO,Avant Interventional Psychiatry,Okah Anyokwu,Psychiatrist,7,"2014, 2015, 2016, 2017, 2023, 2024, 2025",NaN,okah anyokwu,avant interventional psychiatry,united states,4,4,3,1,3.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8015,United States,MA,Sage/Biogen,Sarah Martini,NaN,1,2023,NaN,sarah martini,sage/biogen,united states,1,2,1,1,1.45
8016,United States,MA,Sage/Biogen,Sean Murphy,NaN,1,2023,NaN,sean murphy,sage/biogen,united states,1,2,1,1,1.45
8017,United States,MA,Sage/Biogen,Sibin Stephen,Senior Medical Science Liaison,1,2023,NaN,sibin stephen,sage/biogen,united states,1,2,1,1,1.45
8018,United States,MA,Sage/Biogen,Sophia Yu,"Associate Director, Marketing",1,2023,NaN,sophia yu,sage/biogen,united states,1,2,1,5,2.25


In [65]:
df = df.sort_values(by='FinalPreScore',ascending=False)
df

,Country,State,Institution,Name,Title,Appearances,Years,Unnamed: 7,Name_clean,Institution_clean,Country_clean,FrequencyScore,RecencyScore,MomentumScore,AuthorityScore,FinalPreScore
18149,United States,SC,33-Washington State Psychiatric Association,James Lee,Medical Director,8,"2014, 2017, 2023, 2023, 2024, 2024, 2025, 2025",NaN,james lee,33-washington state psychiatric association,united states,5,4,3,5,4.35
12995,United States,NY,Billion Minds Institute,Gary Belkin,Director,5,"2014, 2017, 2023, 2024, 2025",NaN,gary belkin,billion minds institute,united states,4,4,3,5,4.10
10739,United States,NE,DISTINGGUISHED LIFE FELLOW,Sanat Roy,medical director,7,"2014, 2015, 2016, 2017, 2023, 2024, 2025",NaN,sanat roy,distingguished life fellow,united states,4,4,3,5,4.10
5193,United States,GA,11-Georgia Psychiatric Physicians Association Inc,Edward Ajayi,Medical Director,7,"2014, 2015, 2016, 2017, 2023, 2024, 2025",NaN,edward ajayi,11-georgia psychiatric physicians association inc,united states,4,4,3,5,4.10
12623,United States,NY,33-Washington State Psychiatric Association,Matthew Goldman,"Medical Director, Comprehensive Crisis Services",5,"2015, 2017, 2023, 2024, 2025",NaN,matthew goldman,33-washington state psychiatric association,united states,4,4,3,5,4.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16746,United States,OK,NaN,Fady Fayek,PSYCHIATRY RESIDENT,1,2016,NaN,fady fayek,NaN,united states,1,0,0,1,0.45
16747,United States,OK,NaN,Irina Baranskaya,NaN,1,2014,NaN,irina baranskaya,NaN,united states,1,0,0,1,0.45
16748,United States,OK,NaN,J. Bryan Cates,NaN,1,2017,NaN,j. bryan cates,NaN,united states,1,0,0,1,0.45
16749,United States,OK,NaN,Jennifer Brady,NaN,1,2017,NaN,jennifer brady,NaN,united states,1,0,0,1,0.45


In [69]:
# -----------------------
# FILTER TOP 20%
# -----------------------

threshold = df["FinalPreScore"].quantile(0.80)

df_top20 = df[df["FinalPreScore"] >= threshold]

print("Top 20% size:", len(df_top20))


# -----------------------
# EXPORT FOR CLAY
# -----------------------

df_top20[[
    "Name",
    "Institution",
    "Title",
    "Country",
    "State",
    "Appearances",
    "Years",
    "FrequencyScore",
    "RecencyScore",
    "MomentumScore",
    "AuthorityScore",
    "FinalPreScore"
]].to_csv("top20_for_clay.csv", index=False)

print("Export complete.")

Top 20% size: 4077
Export complete.
